# Phase 2: Grounded-SAM masks + hint-based colorization (Colab)

Consumes the 5 hand-authored plans in `examples/plans/phase2/`:
1. **Grounded-SAM**: each region's `grounding_phrase` → Grounding DINO box → SAM mask. Uses HuggingFace `transformers` ports (no CUDA-extension compilation).
2. **Naive render + adherence** (sanity check, runs immediately).
3. **Control Color**: hint image (strokes from masks) → diffusion colorizer. Scaffolded; needs live iteration like L-CAD did.

Grounding runs on the **grayscale** image — the honest test-time input.

> Runtime → GPU before running.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/tomqi6195/chroma-reasoner.git
%cd /content
!pip install -q -e chroma-reasoner
!pip install -q pycocotools
# Pin transformers<5 for the WHOLE session up front: Control Color (section 4)
# requires 4.x, and downgrading mid-session after section 2 has already
# imported 5.x leaves a broken mixed state that forces a kernel restart.
# Grounding DINO + SAM (section 2) work fine on 4.x.
!pip install -q "transformers<5"
# restart-free editable install: make the package importable now
import sys; sys.path.insert(0, '/content/chroma-reasoner/src')
import transformers; print('transformers', transformers.__version__)

## 1. Fetch the 5 images + make grayscale

In [ ]:
import glob, json, os, urllib.request
import cv2

PLANS = sorted(glob.glob('chroma-reasoner/examples/plans/phase2/*.json'))
IMAGE_IDS = [json.load(open(p))['image_id'] for p in PLANS]
print(IMAGE_IDS)

os.makedirs('images', exist_ok=True); os.makedirs('gray', exist_ok=True)
for iid in IMAGE_IDS:
    dest = f'images/{iid}.jpg'
    if not os.path.exists(dest):
        urllib.request.urlretrieve(f'http://images.cocodataset.org/val2017/{iid}.jpg', dest)
    img = cv2.imread(dest)
    lab_l = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[:, :, 0]
    cv2.imwrite(f'gray/{iid}.png', lab_l)
print('done')

## 2. Grounded-SAM: grounding_phrase → box → mask

In [ ]:
import numpy as np
import torch
from PIL import Image
from transformers import (AutoModelForZeroShotObjectDetection, AutoProcessor,
                          SamModel, SamProcessor)

device = 'cuda'
dino_id = 'IDEA-Research/grounding-dino-base'
sam_id = 'facebook/sam-vit-huge'   # drop to sam-vit-base if VRAM-tight
dino_proc = AutoProcessor.from_pretrained(dino_id)
dino = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device).eval()
sam_proc = SamProcessor.from_pretrained(sam_id)
sam = SamModel.from_pretrained(sam_id).to(device).eval()

@torch.no_grad()
def phrase_to_mask(image_pil, phrase):
    # Grounding DINO wants lowercase queries terminated with '.'
    text = phrase.lower().rstrip('.') + '.'
    inputs = dino_proc(images=image_pil, text=text, return_tensors='pt').to(device)
    out = dino(**inputs)
    res = dino_proc.post_process_grounded_object_detection(
        out, inputs.input_ids, threshold=0.25, text_threshold=0.2,
        target_sizes=[image_pil.size[::-1]])[0]
    if len(res['boxes']) == 0:
        return None, 0.0
    best = res['scores'].argmax()
    box = res['boxes'][best].tolist()
    s_in = sam_proc(image_pil, input_boxes=[[box]], return_tensors='pt').to(device)
    s_out = sam(**s_in)
    masks = sam_proc.image_processor.post_process_masks(
        s_out.pred_masks.cpu(), s_in['original_sizes'].cpu(), s_in['reshaped_input_sizes'].cpu())[0][0]
    scores = s_out.iou_scores.cpu()[0, 0]
    return masks[scores.argmax()].numpy().astype(bool), float(res['scores'][best])

In [ ]:
from chroma_reasoner.plan import load_plan
from chroma_reasoner.plan.masks import region_key, save_mask

MASKS_ROOT = 'masks'
for plan_path in PLANS:
    plan = load_plan(plan_path)
    iid = plan['image_id']
    # ground on the grayscale image, replicated to 3 channels
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    pil = Image.fromarray(np.stack([gray] * 3, axis=-1))
    for region in plan['regions']:
        mask, score = phrase_to_mask(pil, region['grounding_phrase'])
        if mask is None:
            print(f"!! {iid}/{region_key(region)}: NO DETECTION - simplify the phrase")
            continue
        save_mask(mask, MASKS_ROOT, iid, region)
        print(f"{iid}/{region_key(region)}: dino score {score:.2f}, mask {mask.sum()} px")

In [ ]:
# Visual check: overlay every mask on its image. Wrong masks => fix phrases
# in the plan JSON and re-run the cell above.
import matplotlib.pyplot as plt

for plan_path in PLANS:
    plan = load_plan(plan_path)
    iid = plan['image_id']
    img = cv2.cvtColor(cv2.imread(f'images/{iid}.jpg'), cv2.COLOR_BGR2RGB)
    regions = plan['regions']
    fig, axes = plt.subplots(1, len(regions) + 1, figsize=(4 * (len(regions) + 1), 4))
    axes[0].imshow(img); axes[0].set_title(iid); axes[0].axis('off')
    for ax, region in zip(axes[1:], regions):
        key = region_key(region)
        path = f'{MASKS_ROOT}/{iid}/{key}.png'
        ax.axis('off'); ax.set_title(key)
        if os.path.exists(path):
            m = cv2.imread(path, cv2.IMREAD_GRAYSCALE) > 127
            over = img.copy(); over[m] = (0.4 * over[m] + 0.6 * np.array([255, 0, 0])).astype(np.uint8)
            ax.imshow(over)
    plt.show()

## 3. Naive render + adherence (instant end-to-end check)

In [ ]:
from chroma_reasoner.plan.adherence import evaluate_adherence
from chroma_reasoner.plan.hints import render_naive
from chroma_reasoner.plan.masks import load_masks

os.makedirs('results/naive', exist_ok=True)
for plan_path in PLANS:
    plan = load_plan(plan_path)
    iid = plan['image_id']
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    try:
        masks = load_masks(MASKS_ROOT, iid, plan, shape=gray.shape)
    except FileNotFoundError as e:
        print(f'{iid}: {e}'); continue
    rgb = render_naive(gray, masks, plan)
    cv2.imwrite(f'results/naive/{iid}.png', cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    report = evaluate_adherence(rgb, masks, plan)
    print(iid, '| mean dE', report['mean_delta_e'], '| pass', report['n_pass'], '/', report['n_regions'])
    plt.figure(figsize=(6, 6)); plt.imshow(rgb); plt.axis('off'); plt.show()

## 4. Control Color (diffusion colorizer)

Interface (from reading `test.py`): `process(...)` takes `input_image` (grayscale, 3-channel), `hint_image` (same image with colour strokes painted on), and a text `prompt`; it recovers the hinted pixels by comparing the two. Our `make_hint_image` builds exactly that from masks + plan.

Checkpoints (`main_model.ckpt`, `content-guided_deformable_vae.ckpt`) come from the Google Drive linked in the repo README → `pretrained_models/`. **Expect the Drive quota problem again** — use the same Make-a-copy workaround as L-CAD if gdown fails.

Like L-CAD, this repo predates today's Colab; expect to iterate on imports/versions on first run (`transformers<5` will likely be needed again for its CLIP).

In [ ]:
%cd /content
!test -d Control-Color || git clone https://github.com/ZhexinLiang/Control-Color.git

# Same LDM-era stack as L-CAD, same transformers<5 requirement: Control Color
# loads its ckpt with strict=False, so a transformers-5 CLIP key mismatch would
# be SILENT (random text encoder), not a crash.
!pip install -q pytorch-lightning==1.9.5 omegaconf einops kornia open-clip-torch taming-transformers-rom1504 "transformers<5" gradio safetensors torchviz

!mkdir -p Control-Color/pretrained_models
!gdown --folder https://drive.google.com/drive/folders/1lgqstNwrMCzymowRsbGM-4hk0-7L-eOT -O Control-Color/pretrained_models --remaining-ok
!ls -lh Control-Color/pretrained_models

In [ ]:
# Drive fallback if gdown above hit the quota block: in your browser, open the
# checkpoint folder link, Make a copy of BOTH files into My Drive, then run this.
import glob, os, shutil
from google.colab import drive

need = ['main_model.ckpt', 'content-guided_deformable_vae.ckpt']
if not all(os.path.exists(f'Control-Color/pretrained_models/{n}') for n in need):
    drive.mount('/content/drive')
    for name in need:
        dest = f'Control-Color/pretrained_models/{name}'
        if not os.path.exists(dest):
            hits = glob.glob(f'/content/drive/MyDrive/**/*{name}', recursive=True)
            print(name, '->', hits)
            assert hits, f'no copy of {name} in your Drive yet'
            shutil.copy(hits[0], dest)
!ls -lh Control-Color/pretrained_models

In [ ]:
# Build hint images for every plan (works now, independent of Control Color).
from chroma_reasoner.plan.hints import make_hint_image

os.makedirs('hints', exist_ok=True)
for plan_path in PLANS:
    plan = load_plan(plan_path)
    iid = plan['image_id']
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    gray_rgb = np.stack([gray] * 3, axis=-1)
    masks = load_masks(MASKS_ROOT, iid, plan, shape=gray.shape)
    hint = make_hint_image(gray_rgb, masks, plan, erosion=0.15)
    cv2.imwrite(f'hints/{iid}_input.png', cv2.cvtColor(gray_rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(f'hints/{iid}_hint.png', cv2.cvtColor(hint, cv2.COLOR_RGB2BGR))
    plt.figure(figsize=(6, 6)); plt.imshow(hint); plt.axis('off'); plt.title(f'{iid} hint'); plt.show()

# TODO(first run): adapt Control-Color/test.py — extract its model setup and
# call process(change_according_to_strokes=True, iterative_editing=False,
#              input_image=<hints/{iid}_input.png>, hint_image=<hints/{iid}_hint.png>, ...)
# then save outputs to results/ctrlcolor/{iid}.png and run evaluate_adherence on them.

In [ ]:
# Their test.py loads the model then launches a Gradio UI at module level.
# Keep the model/setup part, drop the UI, import process() from the result.
#
# lavis (BLIP) is only used to auto-caption when prompt==''; we always pass a
# prompt, so stub the module instead of installing 2023-era LAVIS on py3.12.
import sys, types
fake_models = types.ModuleType('lavis.models')
def load_model_and_preprocess(name=None, model_type=None, is_eval=True, device=None):
    return None, {'eval': None}, None
fake_models.load_model_and_preprocess = load_model_and_preprocess
fake = types.ModuleType('lavis'); fake.models = fake_models
sys.modules['lavis'] = fake; sys.modules['lavis.models'] = fake_models

src = open('/content/Control-Color/test.py').read()
cut = src.find('block = gr.Blocks')
if cut < 0:
    cut = src.find('gr.Blocks')
assert cut > 0, "couldn't find the Gradio section - inspect test.py manually"
open('/content/Control-Color/test_api.py', 'w').write(src[:cut])

%cd /content/Control-Color
if '.' not in sys.path:
    sys.path.insert(0, '.')
from test_api import process   # loads main_model.ckpt onto the GPU - takes a minute
%cd /content
print('process() ready')

In [ ]:
import cv2, numpy as np, os
import matplotlib.pyplot as plt
from chroma_reasoner.plan import load_plan
from chroma_reasoner.plan.adherence import evaluate_adherence
from chroma_reasoner.plan.masks import load_masks

# BLIP is stubbed out, so every image MUST get a non-empty prompt.
PROMPTS = {
    '000000000139': 'a woman standing in a living room with a television and dining table',
    '000000001000': 'a group of children posing on a tennis court',
    '000000002299': 'a class photo of school children in front of a stone wall',
    '000000010092': 'a bedroom with a mosquito net over the bed',
    '000000022755': 'a school bus reflected in a round mirror',
}

os.makedirs('/content/results/ctrlcolor', exist_ok=True)
%cd /content/Control-Color

for plan_path in PLANS:
    plan = load_plan(f'/content/{plan_path}')
    iid = plan['image_id']
    inp = cv2.cvtColor(cv2.imread(f'/content/hints/{iid}_input.png'), cv2.COLOR_BGR2RGB)
    hint = cv2.cvtColor(cv2.imread(f'/content/hints/{iid}_hint.png'), cv2.COLOR_BGR2RGB)
    outs = process(
        using_deformable_vae=False,        # True needs the VAE ckpt; try after first success
        change_according_to_strokes=True,
        iterative_editing=False,
        input_image=inp, hint_image=hint,
        prompt=PROMPTS[iid], a_prompt='best quality, detailed, real',
        n_prompt='a black and white photo, longbody, lowres, bad anatomy, extra digit, cropped, worst quality, low quality',
        num_samples=1, image_resolution=512, ddim_steps=20,
        guess_mode=False, strength=1.0, scale=7.0,
        sag_scale=0.05, SAG_influence_step=600, seed=42, eta=0.0)
    out = outs[0] if isinstance(outs, list) else outs
    # process() works at 512; restore original size for adherence vs full-res masks
    H, W = inp.shape[:2]
    out_full = cv2.resize(out, (W, H), interpolation=cv2.INTER_LANCZOS4)
    cv2.imwrite(f'/content/results/ctrlcolor/{iid}.png', cv2.cvtColor(out_full, cv2.COLOR_RGB2BGR))

    gray = cv2.imread(f'/content/gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    masks = load_masks('/content/masks', iid, plan, shape=gray.shape)
    report = evaluate_adherence(out_full, masks, plan)
    print(iid, '| mean dE', report['mean_delta_e'], '| pass', report['n_pass'], '/', report['n_regions'])
    plt.figure(figsize=(6, 6)); plt.imshow(out_full); plt.axis('off'); plt.title(iid); plt.show()

%cd /content

## 5. Export masks + outputs

In [ ]:
!zip -rq phase2_outputs.zip masks results hints
from google.colab import files
files.download('phase2_outputs.zip')
# Locally: extract into the repo root, then e.g.
#   python scripts/render_naive.py --plan examples/plans/phase2/000000010092.json \
#       --gray data/coco/gray/000000010092.png --masks masks --out results/phase2/naive